In [2]:
import gensim
import os

In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess

nltk.download("stopwords")
nltk.download("punkt")

# Load stopwords ONCE
stop_words = set(stopwords.words("english"))

story = []

# def remove_stopwords(tokens):
#     return [word for word in tokens if word not in stop_words]

for filename in os.listdir("Datasets/friends"):
    path = os.path.join("Datasets/friends", filename)
    
    with open(path, "r", encoding="utf-8") as f:
        corpus = f.read()
    
    raw_sentences = sent_tokenize(corpus)
    for sent in raw_sentences:
        token=simple_preprocess(sent,min_len=1)
        if len(token)>0:
            story.append(token)
     

    #for lstm corpus
    for sent in raw_sentences:
        words = simple_preprocess(sent)
        # words = remove_stopwords(words)
        story.append(words)

print(story[:5])


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\narin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\narin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


[['the', 'one', 'where', 'monica', 'gets', 'a', 'new', 'roommate', 'the', 'pilot', 'the', 'uncut', 'version', 'written', 'by', 'marta', 'kauffman', 'david', 'crane', 'scene', 'central', 'perk', 'chandler', 'joey', 'phoebe', 'and', 'monica', 'are', 'there'], ['monica', 'there', 's', 'nothing', 'to', 'tell'], ['he', 's', 'just', 'some', 'guy', 'i', 'work', 'with'], ['joey', 'c', 'mon', 'you', 're', 'going', 'out', 'with', 'the', 'guy'], ['there', 's', 'gotta', 'be', 'something', 'wrong', 'with', 'him']]


# lstm specific(beginnig)

In [4]:
import re
DATA_DIR = "Datasets/friends"

SPEAKER_REGEX = re.compile(r"^([A-Z][A-Z ]+):")
PUNCT_REGEX = re.compile(r"([.!?,])")

tokens = []

for filename in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, filename)

    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Scene descriptions
        if line.startswith("(") or line.startswith("[") or line.isupper():
            tokens.append("<SCENE>")
            continue

        # Speaker line
        speaker_match = SPEAKER_REGEX.match(line)
        if speaker_match:
            speaker = speaker_match.group(1).replace(" ", "_")
            tokens.append(f"<{speaker}>")
            line = line[speaker_match.end():].strip()

        # Normalize text
        line = line.lower()

        # Separate punctuation
        line = PUNCT_REGEX.sub(r" \1 ", line)

        # Tokenize
        words = line.split()
        tokens.extend(words)


In [6]:
from gensim.models import Word2Vec
model=Word2Vec(
    window=7,min_count=2,compute_loss=True,vector_size=128,sg=1,hs=1,workers=16
)

In [ ]:
model.build_vocab(story)

print(model.wv.index_to_key)     
print(len(model.wv))


['you', 'the', 'to', 'and', 'i', 'it', 'ross', 'rachel', 'joey', 'chandler', 'monica', 'that', 'phoebe', 'is', 's', 'what', 'oh', 'a', 'in', 'know', 'of', 'no', 'we', 'this', 'on', 'just', 'he', 'me', 'so', 'my', 'with', 'she', 'her', 't', 'yeah', 'are', 'okay', 're', 'do', 'for', 'have', 'not', 'all', 'don', 'well', 'can', 'hey', 'was', 'up', 'but', 'at', 'be', 'out', 'they', 'right', 'there', 'like', 'your', 'scene', 'm', 'get', 'him', 'about', 'his', 'gonna', 'go', 'one', 'here', 'uh', 'really', 'look', 'if', 'think', 'how', 'll', 'now', 'see', 'from', 'mean', 'back', 'did', 'got', 'as', 'good', 'why', 'come', 'want', 'then', 'god', 'who', 've', 'would', 'when', 'guys', 'going', 'over', 'sorry', 'time', 'hi', 'down', 'little', 'ok', 'great', 'some', 'guy', 'let', 'say', 'them', 'tell', 'yes', 'were', 'by', 'didn', 'door', 'an', 'because', 'something', 'y', 'could', 'off', 'room', 'too', 'had', 'again', 'into', 'wait', 'thing', 'love', 'starts', 'looks', 'or', 'has', 'where', 'take',

In [ ]:
print(type(story))
print(type(story[0]))


<class 'list'>
<class 'list'>


In [ ]:
model.train(story,total_examples=model.corpus_count,
            epochs=10)

(3612803, 4689450)

In [ ]:
model.wv.most_similar(
    positive=["carol", "susan"],
    negative=["ross"],
    topn=5
)


[('marlon', 0.5637405514717102),
 ('gyn', 0.5356979370117188),
 ('ob', 0.5266807079315186),
 ('cradled', 0.5123409032821655),
 ('minnie', 0.4604467749595642)]

In [ ]:
model.wv.most_similar(
    positive=["monica", "rachel"],
    negative=["ross"],
    topn=5
)


[('chandler', 0.7253305315971375),
 ('phoebe', 0.7239635586738586),
 ('joey', 0.600207507610321),
 ('waiters', 0.5911075472831726),
 ('min', 0.5732181072235107)]

In [ ]:
model.wv.most_similar(
    positive=["ross", "emma"],
    negative=["rachel"],
    topn=5
)


[('demma', 0.5246222615242004),
 ('wemma', 0.5083251595497131),
 ('attending', 0.4604671597480774),
 ('mmmwa', 0.4556904435157776),
 ('anticipation', 0.45359113812446594)]

In [ ]:
model.wv.most_similar("apartment")


[('dejectedly', 0.5993455648422241),
 ('pacing', 0.5940915942192078),
 ('bedroom', 0.5877516269683838),
 ('launderama', 0.5830675363540649),
 ('hallway', 0.5813760161399841),
 ('lang', 0.5795578956604004),
 ('hall', 0.5783059597015381),
 ('rachels', 0.575838565826416),
 ('philly', 0.5711270570755005),
 ('confront', 0.5676891207695007)]

In [ ]:
model.wv.doesnt_match(["ross", "rachel", "chandler", "joey"])


'joey'

In [ ]:
model.wv.doesnt_match(["central", "perk", "coffee", "ross"])


'coffee'

In [ ]:
pairs = [
    ("ross","rachel"),
    ("monica","chandler"),
    ("joey","phoebe"),
    ("ross","chandler"),
    ("ross","pizza"),
    ("monica","coffee")
]

for a,b in pairs:
    print(a, b, model.wv.similarity(a,b))


ross rachel 0.8704301
monica chandler 0.82290894
joey phoebe 0.71578276
ross chandler 0.8080129
ross pizza 0.29300457
monica coffee 0.26570573


In [ ]:
tests = [
    ("ross","rachel","monica","chandler"),
    ("monica","chandler","ross","rachel"),
    ("woman","man","queen","king")
]

correct = 0
for a,b,c,d in tests:
    pred = model.wv.most_similar(positive=[a,c], negative=[b], topn=1)[0][0]
    print(a,":",b,"::",c,":",pred)
    if pred == d:
        correct += 1

print("Accuracy:", correct/len(tests))


ross : rachel :: monica : chandler
monica : chandler :: ross : rachel
woman : man :: queen : steele
Accuracy: 0.6666666666666666
